In [0]:
from pyspark.sql.functions import col, lit, year, month, date_format, greatest, least, datediff
from datetime import date

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df_bronze = spark.table("scottish_housing_ws.bronze.festival_dates")
df_bronze.show(30, truncate=False)

In [0]:
import pandas as pd
from datetime import date

months = []
for y in range(2004, 2027):
    for m in [7, 8]:  # July and August only -- Fringe never extends beyond these
        months.append({
            "year": y,
            "month_num": m,
            "month_start": date(y, m, 1),
            # Last day of the month
            "month_end": date(y, m, 31) if m == 8 else date(y, m, 31 if m in [1,3,5,7,10,12] else 30)
        })

df_spine = spark.createDataFrame(pd.DataFrame(months))
df_spine.show(10)

In [0]:
df_joined = df_spine.join(df_bronze, on="year", how="left")

df_silver = (
    df_joined
    .withColumn(
        "overlap_start",
        greatest(col("month_start"), col("fringe_start"))
    )
    .withColumn(
        "overlap_end",
        least(col("month_end"), col("fringe_end"))
    )
    .withColumn(
        "festival_days_in_month",
        greatest(
            lit(0),
            datediff(col("overlap_end"), col("overlap_start")) + lit(1)
        )
    )
    .withColumn(
        "is_festival_month",
        col("festival_days_in_month") > 0
    )
    .withColumn(
        "year_month",
        date_format(col("month_start"), "yyyy-MM")
    )
    .withColumnRenamed("month_start", "report_date")
    .withColumnRenamed("cancelled", "is_cancelled")
    .select(
        "report_date",
        "year_month",
        "year",
        "fringe_start",
        "fringe_end",
        "festival_days_in_month",
        "is_festival_month",
        "is_cancelled"
    )
    .orderBy("report_date")
)

df_silver.printSchema()

In [0]:
from pyspark.sql.functions import count
df_silver.show(46, truncate=False)
df_silver.agg(count("*").alias("row_count")).show()

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.festival_dates")
)

In [0]:
# One row per month (July and August only) per year, 2004 to 2026.
# festival_days_in_month counts the overlap between the Fringe dates and that calendar month.
# is_festival_month is true wherever festival_days_in_month > 0.
# is_cancelled=True for 2020 -- festival ran online only, no city impact.
# 46 rows total: 23 years x 2 months.